# blender_pipeline_v2 — تست Integration روی GPU

این notebook تست کامل pipeline را روی GPU واقعی انجام می‌دهد:
1. Clone ریپو از GitHub
2. نصب TripoSR و وابستگی‌ها
3. ساخت تصویر تست (بدون نیاز به دانلود)
4. اجرای pipeline
5. بررسی خروجی: mesh.obj و scene.py

> **مهم:** Runtime → Change runtime type → **T4 GPU** را انتخاب کن.

## سلول ۰ — بررسی GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: GPU ندارد — pipeline با device=auto روی CPU اجرا می‌شود (کندتر)')

## سلول ۱ — Clone ریپو

In [ ]:
!git clone https://github.com/nasrialireza959-pixel/blender_pipeline.git
%cd blender_pipeline

## سلول ۲ — نصب وابستگی‌ها و TripoSR

In [ ]:
!pip install -q rembg trimesh omegaconf pyyaml pillow
!python scripts/setup.py
print('نصب تمام شد.')

## سلول ۳ — ساخت تصویر تست

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

img_path = Path('examples/sample_images/test_input.jpg')
img_path.parent.mkdir(parents=True, exist_ok=True)

# ساخت تصویر مصنوعی: یک کره قرمز روی پس‌زمینه سفید
size = 512
img_array = np.ones((size, size, 3), dtype=np.uint8) * 255
cx, cy, r = size // 2, size // 2, size // 3
y_grid, x_grid = np.ogrid[:size, :size]
dist = np.sqrt((x_grid - cx) ** 2 + (y_grid - cy) ** 2)
mask = dist <= r
# رنگ با shading ساده
shade = np.clip(1.0 - (dist / r) * 0.4, 0.6, 1.0)
img_array[mask, 0] = (shade[mask] * 200).astype(np.uint8)
img_array[mask, 1] = (shade[mask] * 60).astype(np.uint8)
img_array[mask, 2] = (shade[mask] * 60).astype(np.uint8)

img = Image.fromarray(img_array)
img.save(img_path, quality=95)

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title('تصویر تست (کره مصنوعی)')
plt.axis('off')
plt.show()
print('تصویر ساخته شد:', img_path, img.size)

## سلول ۴ — اجرای pipeline

In [ ]:
import sys
sys.path.insert(0, 'src')

from core.config import load_config
from orchestration.pipeline import Pipeline

config = load_config('configs/default.yaml')
print('device config:', config.triposr.device)  # باید 'auto' باشد

pipeline = Pipeline(config)
print('\nشروع pipeline...')
job = pipeline.run(str(img_path), object_name='TestSphere')

print(f'\nوضعیت: {job.status}')
print(f'Job ID: {job.id}  (length={len(job.id)})')
print(f'mesh: {job.mesh_path}')
print(f'script: {job.blender_script_path}')

## سلول ۵ — بررسی خروجی mesh

In [ ]:
import trimesh

mesh = trimesh.load(str(job.mesh_path))
print('تعداد vertices:', len(mesh.vertices))
print('تعداد faces:', len(mesh.faces))
print('اندازه فایل:', round(job.mesh_path.stat().st_size / 1024, 1), 'KB')

assert len(mesh.vertices) > 0, 'mesh خالی است!'
assert len(mesh.faces) > 0, 'mesh face ندارد!'
print('\nمش سالم است.')

## سلول ۶ — بررسی اسکریپت Blender

In [ ]:
script = job.blender_script_path.read_text(encoding='utf-8')

assert 'bpy.ops.wm.obj_import' in script, 'API مدرن Blender 4.x نیست!'
assert 'import_scene.obj' not in script, 'API منسوخ پیدا شد!'
assert 'TestSphere' in script, 'نام object در اسکریپت نیست!'

print('اسکریپت Blender تایید شد.')
print('\n--- ۲۰ خط اول اسکریپت ---')
print('\n'.join(script.splitlines()[:20]))

## سلول ۷ — بررسی job.json

In [ ]:
import json

metadata = json.loads(job.metadata_path.read_text(encoding='utf-8'))
print(json.dumps(metadata, ensure_ascii=False, indent=2))

assert metadata['status'] == 'done', f"وضعیت اشتباه: {metadata['status']}"
assert metadata['error'] is None, f"خطا ثبت شده: {metadata['error']}"
assert len(metadata['id']) == 12, f"UUID باید ۱۲ کاراکتر باشد: {metadata['id']}"
print('\nجمع‌بندی: pipeline با موفقیت اجرا شد.')

## سلول ۸ — اجرای unit tests (تایید نهایی)

In [ ]:
!python -m pytest tests/ -v